# 🔧 Tool Calling Tutorial: Fine-Tuning and Inference

#### 📚 What you'll learn

This notebook covers LoRA fine-tuning and inference testing with Nemotron 3 Nano:

- How to create a customization target for Nemotron 3 Nano
- How to configure and launch a LoRA fine-tuning job using the SDK's typed constructors
- How to monitor training progress
- How to run inference with the fine-tuned model

Start with [Notebook 1: Data Preparation](./1_data_preparation.ipynb) if you haven't prepared your data.


### 📦 Imports

- `nemo_microservices` provides the NMP platform client and typed parameter classes.
- `openai` provides the OpenAI-compatible client for NIM inference.


### ⚡ Prerequisites and Setup

Before running this notebook, you need:

1. **A running NMP deployment** with NeMo Customizer enabled. See [platform setup](https://docs.nvidia.com/nemo/microservices/latest/get-started/setup/index.html).
2. **A Nemotron 3 Nano NIM deployed** for inference. Deploy via the [NIM deployment tutorial](https://docs.nvidia.com/nemo/microservices/latest/get-started/tutorials/deploy-nims.html). Deployment takes ~10 minutes.
3. **Completed [Notebook 1](./1_data_preparation.ipynb)** (or [Notebook 2](./2_data_designer.ipynb)) to prepare training data in filesets.


In [ ]:
%%capture
!pip install -r requirements.txt

In [ ]:
import os
import json
import random
from time import sleep, time

from openai import OpenAI
from nemo_microservices import NeMoMicroservices
from nemo_microservices.types.customization import (
    CustomizationJobInputParam,
    HyperparametersParam,
    LoRaParamsParam,
)

from config import (
    NEMO_URL, NIM_URL, WORKSPACE, BASE_MODEL, BASE_MODEL_URI,
    TARGET_NAME, JOB_NAME, DD_TRAINING_FILESET, TRAINING_FILESET,
    WANDB_API_KEY,
)

### ⚙️ Initialize the NeMo Microservices client


In [ ]:
client = NeMoMicroservices(
    base_url=NEMO_URL,
    inference_base_url=NIM_URL,
    workspace=WORKSPACE,
)

## 🎯 Create a Customization Target

- A customization target registers a base model for fine-tuning.
- We use Nemotron 3 Nano, which supports LoRA and full-weight fine-tuning.

> 💡 **Nemotron 3 Nano vs Super**
>
> - **Nemotron 3 Nano**: 30B MoE (3B active parameters). Fast, efficient, great for tutorials.
> - **Nemotron Super** (49B): Higher quality, LoRA only. Best for production use cases requiring maximum accuracy.


In [ ]:
try:
    target = client.customization.targets.create(
        model_uri=BASE_MODEL_URI,
        name=TARGET_NAME,
        description="Nemotron 3 Nano target for tool calling fine-tuning",
    )
    print(f"Created target: {target.name}")
except Exception as e:
    if "409" in str(e):
        print(f"Target {TARGET_NAME} already exists")
        target = client.customization.targets.retrieve(name=TARGET_NAME)
    else:
        raise

## 🏗️ Configure the Fine-Tuning Job

- We build the job config using the SDK's typed constructors for IDE autocompletion.
- Each config piece is a named variable: `lora_config` → `hyperparameters` → `spec`.
- The dataset references the fileset from Notebook 2 (Data Designer output). You can also use `TRAINING_FILESET` for the xLAM data from Notebook 1.


In [ ]:
lora_config = LoRaParamsParam(
    rank=32,
    alpha=64,
    dropout=0.1,
)

In [ ]:
hyperparameters = HyperparametersParam(
    finetuning_type="lora",
    training_type="sft",
    epochs=2,
    batch_size=16,
    micro_batch_size=4,
    learning_rate=0.0001,
    max_seq_length=2048,
    sequence_packing_enabled=True,
    lora=lora_config,
)

In [ ]:
spec = CustomizationJobInputParam(
    target=f"{WORKSPACE}/{TARGET_NAME}",
    dataset=f"fileset://{WORKSPACE}/{DD_TRAINING_FILESET}",
    hyperparameters=hyperparameters,
)

## 🚀 Launch Fine-Tuning

- Submit the job to the Customizer service.
- Training takes approximately 30-60 minutes depending on dataset size.


In [ ]:
nemo_client = client
if WANDB_API_KEY:
    nemo_client = client.with_options(default_headers={"wandb-api-key": WANDB_API_KEY})

job = nemo_client.customization.jobs.create(
    name=JOB_NAME,
    spec=spec,
)

print(f"Launched fine-tuning job: {job.name}")

## ⏳ Monitor Training

- Poll the job status until training completes.


In [ ]:
start = time()
while True:
    status = client.customization.jobs.get_status(name=job.name)
    print(f"Status: {status.status} ({time() - start:.0f}s)")
    if status.status in ["completed", "error", "cancelled"]:
        break
    sleep(30)

sleep(120)  # Wait for NIM to pick up the new adapter
print("Fine-tuning complete!")

### ✅ Verify model availability

- The fine-tuned model (LoRA adapter) is automatically registered and picked up by NIM.


In [ ]:
job_detail = client.customization.jobs.retrieve(name=job.name)
CUSTOMIZED_MODEL = job_detail.spec.output_model
print(f"Customized model: {CUSTOMIZED_MODEL}")

models = client.inference.models.list()
model_names = [m.id for m in models.data]
print(f"Available models: {model_names}")
assert CUSTOMIZED_MODEL in model_names, f"Model {CUSTOMIZED_MODEL} not found in NIM"

## 🧪 Test Inference

- The fine-tuned model is served via NIM with an OpenAI-compatible API.
- Let's test it with a tool calling prompt.


In [ ]:
test_messages = [
    {"role": "user", "content": "What's the weather like in San Francisco today?"}
]

test_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name"},
                    "unit": {"type": "string", "description": "Temperature unit (celsius or fahrenheit)"}
                },
                "required": ["location"]
            },
        },
    }
]

In [ ]:
nim_client = OpenAI(base_url=f"{NIM_URL}/v1", api_key="not-used")

completion = nim_client.chat.completions.create(
    model=CUSTOMIZED_MODEL,
    messages=test_messages,
    tools=test_tools,
    tool_choice="auto",
    temperature=0.1,
    max_tokens=512,
)

print("Tool calls from fine-tuned model:")
for tc in completion.choices[0].message.tool_calls or []:
    print(f"  {tc.function.name}({tc.function.arguments})")

### 📝 Note your customized model name

- You'll need this in the next notebook for evaluation.


In [ ]:
print(f"Your customized model: {CUSTOMIZED_MODEL}")

## ⏭️ Next Steps

The model is fine-tuned and responding with tool calls. In the next notebook, we'll rigorously evaluate the improvement.

- [4. Model Evaluation](./4_model_evaluation.ipynb)
